# Analisis Angket Penggunaan Media (QN + QL)

Notebook ini mengolah data angket penggunaan media Bicaranta dari kelompok eksperimen
sesuai metodologi Bab III (analisis deskriptif, reliabilitas, korelasi, persentase respons).

**Cakupan:**
1. Validasi data QN (Q1–Q15 skala Likert 1–4)
2. Statistik deskriptif per dimensi dan total
3. Alpha Cronbach per dimensi dan keseluruhan
4. Distribusi respons per item (profil)
5. Persentase skor (rumus P = ΣR/N × 100%) dan kategori
6. Korelasi Pearson skor media ↔ nilai postes berbicara
7. Ringkasan respons kualitatif (QL)


In [1]:
import csv, math
from pathlib import Path
from collections import Counter
from statistics import mean, median, stdev, variance

def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data' / 'QN_angket_penggunaan_media.csv').exists():
            return p
    raise FileNotFoundError("Root proyek tidak ditemukan")

ROOT     = find_root()
QN_PATH  = ROOT / 'data' / 'QN_angket_penggunaan_media.csv'
QL_PATH  = ROOT / 'data' / 'QL_angket_penggunaan_media.csv'
POST_PATH = ROOT / 'data' / 'field_test' / 'keterampilan_berbicara_postes.csv'

ITEM_COLS = [f'Q{i}' for i in range(1, 16)]
DIM_COLS  = ['skor_planning', 'skor_monitoring', 'skor_evaluation']
DIM_LABEL = {'skor_planning': 'Perencanaan', 'skor_monitoring': 'Pemantauan',
             'skor_evaluation': 'Evaluasi'}
DIMS_ITEMS = {
    'Perencanaan': ['Q1','Q4','Q6','Q8','Q15'],
    'Pemantauan' : ['Q5','Q7','Q9','Q10','Q11'],
    'Evaluasi'   : ['Q2','Q3','Q12','Q13','Q14'],
}

def load_qn():
    with open(QN_PATH, encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    out = []
    for row in rows:
        if row.get('kelompok') != 'eksperimen' or not row.get('Q1'):
            continue
        for col in ITEM_COLS + DIM_COLS + ['skor_total']:
            if row.get(col):
                row[col] = float(row[col])
        out.append(row)
    return out

def load_post():
    with open(POST_PATH, encoding='utf-8-sig', newline='') as f:
        return {r['id']: float(r['post_nilai_akhir']) for r in csv.DictReader(f)}

def load_ql():
    with open(QL_PATH, encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

qn   = load_qn()
post = load_post()
ql   = load_ql()

print(f"Root              : {ROOT}")
print(f"QN eksperimen     : {len(qn)} responden")
ql_eks = [r for r in ql if r.get('id','').startswith('E') and any(r.get(f'ql{i}') for i in range(1,7))]
print(f"QL eksperimen     : {len(ql_eks)} responden dengan jawaban")


Root              : /home/primandhika/artikel/dist
QN eksperimen     : 40 responden
QL eksperimen     : 0 responden dengan jawaban


## 1. Validasi Data QN

In [2]:
issues = []
for row in qn:
    sid = row['id']
    for col in ITEM_COLS:
        v = row.get(col)
        if isinstance(v, float) and not (1 <= v <= 4):
            issues.append(f"{sid} {col}={v} di luar [1,4]")
    if all(row.get(c) for c in DIM_COLS) and row.get('skor_total'):
        calc = round(sum(row[c] for c in DIM_COLS) / 3, 2)
        stored = round(row['skor_total'], 2)
        if abs(calc - stored) > 0.11:
            issues.append(f"{sid}: skor_total={stored} != rerata_dim={calc:.2f}")

if issues:
    print("Isu ditemukan:")
    for i in issues: print(" -", i)
else:
    print(f"PASS: {len(qn)} baris valid — semua item dalam [1,4], skor konsisten")


PASS: 40 baris valid — semua item dalam [1,4], skor konsisten


## 2. Statistik Deskriptif

In [3]:
def describe(x):
    return {'n':len(x),'mean':mean(x),'sd':stdev(x) if len(x)>1 else 0,
            'median':median(x),'min':min(x),'max':max(x)}

def fmt(v, d=3):
    return f"{v:.{d}f}" if isinstance(v, float) else str(v)

def print_table(headers, rows):
    text = [[fmt(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i,h in enumerate(headers)]
    sep = ' | '
    print(sep.join(str(h).ljust(widths[i]) for i,h in enumerate(headers)))
    print('-+-'.join('-'*w for w in widths))
    for row in text:
        print(sep.join(row[i].ljust(widths[i]) for i in range(len(headers))))

desc_rows = []
for col in DIM_COLS + ['skor_total']:
    vals = [row[col] for row in qn if isinstance(row.get(col), float)]
    d = describe(vals)
    label = DIM_LABEL.get(col, 'Total')
    desc_rows.append([label, d['n'], d['mean'], d['sd'], d['median'], d['min'], d['max']])
print_table(['Dimensi','n','Mean','SD','Median','Min','Max'], desc_rows)

print()
cat = Counter(row.get('kategori','') for row in qn)
print("Distribusi kategori skor total:")
order = ['Sangat Tinggi','Tinggi','Sedang','Rendah']
for k in order:
    v = cat.get(k, 0)
    print(f"  {k:<15}: {v:>2}  ({v/len(qn)*100:.1f}%)")


Dimensi     | n  | Mean  | SD    | Median | Min   | Max  
------------+----+-------+-------+--------+-------+------
Perencanaan | 40 | 3.305 | 0.506 | 3.400  | 2.200 | 4.000
Pemantauan  | 40 | 3.080 | 0.394 | 3.000  | 2.200 | 3.800
Evaluasi    | 40 | 3.165 | 0.483 | 3.200  | 2.200 | 4.000
Total       | 40 | 3.184 | 0.425 | 3.270  | 2.200 | 3.870

Distribusi kategori skor total:
  Sangat Tinggi  : 21  (52.5%)
  Tinggi         : 16  (40.0%)
  Sedang         :  0  (0.0%)
  Rendah         :  3  (7.5%)


## 3. Persentase Skor (Rumus Bab III)

**Rumus:** P = (ΣR / N) × 100%  
ΣR = jumlah skor jawaban seluruh responden; N = jumlah skor ideal (jumlah item × skor maks × n responden)


In [4]:
n_resp = len(qn)
skor_maks_item = 4

# Total keseluruhan
sigma_r_total = sum(sum(row[q] for q in ITEM_COLS) for row in qn)
n_ideal_total = len(ITEM_COLS) * skor_maks_item * n_resp
p_total = sigma_r_total / n_ideal_total * 100

print(f"Total (15 item): sigmaR={sigma_r_total:.0f}, N_ideal={n_ideal_total}, P={p_total:.2f}%")
print()

# Per dimensi
print(f"{'Dimensi':<15} {'sigmaR':>8} {'N_ideal':>8} {'P (%)':>7}")
print('-' * 42)
for dim_label, items in DIMS_ITEMS.items():
    sr = sum(sum(row[q] for q in items) for row in qn)
    ni = len(items) * skor_maks_item * n_resp
    p  = sr / ni * 100
    print(f"{dim_label:<15} {sr:>8.0f} {ni:>8} {p:>7.2f}%")


Total (15 item): sigmaR=1494, N_ideal=2400, P=62.25%

Dimensi           sigmaR  N_ideal   P (%)
------------------------------------------
Perencanaan          501      800   62.62%
Pemantauan           510      800   63.75%
Evaluasi             483      800   60.38%


## 4. Alpha Cronbach

In [5]:
def cronbach_alpha(matrix):
    k = len(matrix[0])
    if k < 2: return float('nan')
    item_vars = sum(variance([row[j] for row in matrix]) for j in range(k))
    total_var = variance([sum(row) for row in matrix])
    if total_var == 0: return float('nan')
    return k / (k-1) * (1 - item_vars / total_var)

def interp_alpha(a):
    if a >= 0.9: return "Sangat Baik"
    if a >= 0.8: return "Baik"
    if a >= 0.7: return "Dapat Diterima"
    if a >= 0.6: return "Dipertanyakan"
    return "Buruk"

alpha_rows = []
# Semua 15 item
m_all = [[row[q] for q in ITEM_COLS] for row in qn]
a_all = cronbach_alpha(m_all)
alpha_rows.append(['Total (15 item)', 15, a_all, interp_alpha(a_all)])
# Per dimensi
for dim, items in DIMS_ITEMS.items():
    m = [[row[q] for q in items] for row in qn]
    a = cronbach_alpha(m)
    alpha_rows.append([dim, len(items), a, interp_alpha(a)])

print_table(['Dimensi','k','Alpha Cronbach','Interpretasi'], alpha_rows)


Dimensi         | k  | Alpha Cronbach | Interpretasi
----------------+----+----------------+-------------
Total (15 item) | 15 | -0.081         | Buruk       
Perencanaan     | 5  | 0.156          | Buruk       
Pemantauan      | 5  | -0.186         | Buruk       
Evaluasi        | 5  | -0.169         | Buruk       


## 5. Profil Respons Per Item (% frekuensi pilihan 1–4)

In [6]:
print(f"{'Item':<6}  {'1%':>6} {'2%':>6} {'3%':>6} {'4%':>6}  {'Mean':>6} {'SD':>5}")
print('-' * 52)
for col in ITEM_COLS:
    vals = [row[col] for row in qn if isinstance(row.get(col), float)]
    freq = Counter(int(v) for v in vals)
    n = len(vals)
    m = mean(vals)
    s = stdev(vals) if len(vals)>1 else 0
    print(f"{col:<6}  {freq.get(1,0)/n*100:6.1f} {freq.get(2,0)/n*100:6.1f} "
          f"{freq.get(3,0)/n*100:6.1f} {freq.get(4,0)/n*100:6.1f}  {m:6.3f} {s:5.3f}")


Item        1%     2%     3%     4%    Mean    SD
----------------------------------------------------
Q1        12.5   12.5   25.0   50.0   3.125 1.067
Q2        55.0   25.0   15.0    5.0   1.700 0.911
Q3        45.0   32.5   15.0    7.5   1.850 0.949
Q4         7.5   17.5   32.5   42.5   3.100 0.955
Q5        27.5   30.0   32.5   10.0   2.250 0.981
Q6         7.5    7.5   32.5   52.5   3.300 0.911
Q7        20.0   47.5   25.0    7.5   2.200 0.853
Q8        55.0   40.0    2.5    2.5   1.525 0.679
Q9         5.0   10.0   35.0   50.0   3.300 0.853
Q10       57.5   17.5   20.0    5.0   1.725 0.960
Q11        7.5    5.0   40.0   47.5   3.275 0.877
Q12       37.5   32.5   15.0   15.0   2.075 1.071
Q13        2.5   10.0   45.0   42.5   3.275 0.751
Q14        5.0   12.5   42.5   40.0   3.175 0.844
Q15       65.0   25.0    7.5    2.5   1.475 0.751


## 6. Korelasi Skor Media × Nilai Postes Berbicara

In [7]:
def pearson_r(x, y):
    n = len(x); mx, my = mean(x), mean(y)
    num = sum((a-mx)*(b-my) for a,b in zip(x,y))
    den = math.sqrt(sum((a-mx)**2 for a in x) * sum((b-my)**2 for b in y))
    return num/den if den else 0

def betacf(a,b,x):
    qab,qap,qam=a+b,a+1,a-1; c,d=1.0,max(abs(1-qab*x/qap),3e-300)
    d=1/d; h=d
    for m in range(1,201):
        m2=2*m; aa=m*(b-m)*x/((qam+m2)*(a+m2))
        d=1+aa*d; d=1/max(abs(d),3e-300)*(1 if d>=0 else -1)
        c=1+aa/c; c=max(abs(c),3e-300)*(1 if c>=0 else -1)
        h*=d*c; aa=-(a+m)*(qab+m)*x/((a+m2)*(qap+m2))
        d=1+aa*d; d=1/max(abs(d),3e-300)*(1 if d>=0 else -1)
        c=1+aa/c; c=max(abs(c),3e-300)*(1 if c>=0 else -1)
        delta=d*c; h*=delta
        if abs(delta-1)<3e-14: break
    return h

def ibeta(a,b,x):
    if x<=0: return 0.0
    if x>=1: return 1.0
    bt=math.exp(math.lgamma(a+b)-math.lgamma(a)-math.lgamma(b)+a*math.log(x)+b*math.log1p(-x))
    return bt*betacf(a,b,x)/a if x<(a+1)/(a+b+2) else 1-bt*betacf(b,a,1-x)/b

def p_from_r(r, n):
    if abs(r) >= 1: return 0.0
    t = r * math.sqrt(n-2) / math.sqrt(1-r**2)
    return ibeta((n-2)/2, 0.5, (n-2)/(n-2+t**2))

pairs = [(row['skor_total'], post[row['id']])
         for row in qn if row['id'] in post and isinstance(row.get('skor_total'), float)]
xs, ys = zip(*pairs)
n_pairs = len(pairs)
r_tot = pearson_r(list(xs), list(ys))
p_tot = p_from_r(r_tot, n_pairs)
interp = 'Kuat' if abs(r_tot)>=0.6 else ('Sedang' if abs(r_tot)>=0.4 else 'Lemah')

print(f"Korelasi skor_total media x postes berbicara:")
print(f"  n={n_pairs},  r={r_tot:.3f},  p={p_tot:.4f}  ({interp})")
print()
print("Korelasi per dimensi:")
for col in DIM_COLS:
    pairs_d = [(row[col], post[row['id']])
               for row in qn if row['id'] in post and isinstance(row.get(col), float)]
    if len(pairs_d) < 3: continue
    xd, yd = zip(*pairs_d)
    r = pearson_r(list(xd), list(yd))
    p = p_from_r(r, len(pairs_d))
    label = DIM_LABEL[col]
    print(f"  {label:<12}: r={r:.3f},  p={p:.4f},  n={len(pairs_d)}")


Korelasi skor_total media x postes berbicara:
  n=40,  r=0.214,  p=0.1857  (Lemah)

Korelasi per dimensi:
  Perencanaan : r=0.213,  p=0.1868,  n=40
  Pemantauan  : r=0.200,  p=0.2158,  n=40
  Evaluasi    : r=0.176,  p=0.2784,  n=40


## 7. Ringkasan Respons Kualitatif (QL)

In [8]:
ql_filled = [r for r in ql if r.get('id','').startswith('E') and r.get('aspek_disukai')]
print(f"Responden QL eksperimen: {len(ql_filled)}")
print()

QL_COLS = {
    'aspek_disukai':        'Aspek yang paling disukai',
    'alasan_aspek_disukai': 'Alasan menyukai aspek tsb.',
    'manfaat':              'Manfaat yang dirasakan',
    'kendala':              'Kendala penggunaan',
    'saran_perbaikan':      'Saran perbaikan',
    'rekomendasi':          'Rekomendasi',
}

for col, label in QL_COLS.items():
    resps = [r[col] for r in ql_filled if r.get(col)]
    print(f"=== {label.upper()} (n={len(resps)}) ===")
    for i, resp in enumerate(resps[:3], 1):
        trunc = resp[:110] + '...' if len(resp) > 110 else resp
        print(f"  [{i}] {trunc}")
    print()


 eksperimen: 40

=== ASPEK YANG PALING DISUKAI (n=40) ===
  [1] Fitur interaktif; Teknik Sederhanakan dan Ulangi
  [2] Kualitas materi; Teknik Sederhanakan dan Ulangi
  [3] Kualitas materi

=== ALASAN MENYUKAI ASPEK TSB. (n=40) ===
  [1] penerapan teknik feynman mempermudah pemahaman konsep yg rumit.
  [2] Penerapan teknik sederhana membantu latihan berbicara menjadi lebih terstruktur.
  [3] Materinya ringkas sehingga cepat utk dipelajari.

=== MANFAAT YANG DIRASAKAN (n=40) ===
  [1] Peningkatan keterampilan berbicara; Peningkatan kepercayaan diri; Kesadaran metakognitif; Motivasi belajar
  [2] Peningkatan kepercayaan diri; Peningkatan keterampilan berbicara
  [3] Pemahaman materi yang lebih baik

=== KENDALA PENGGUNAAN (n=40) ===
  [1] Koneksi internet
  [2] Tidak ada kendala
  [3] Koneksi internet; Fitur tidak berfungsi

=== SARAN PERBAIKAN (n=40) ===
  [1] Mohon ditambahkan variasi kuis interaktifnya.
  [2] Pertahankan kesederhanaan materi dan tingkatkan stabilitas tombol navigasi.



---

## ✍️ Generate Catatan Pembahasan Bab IV

Sel berikut menulis file `catatan_pembahasan_*.md` yang dapat langsung digunakan sebagai bahan draf pembahasan Bab IV.


In [9]:
# ── GENERATE: catatan pembahasan Bab IV untuk angket media ──────────────────
import csv, math
from pathlib import Path
from collections import Counter
from statistics import mean, stdev, variance

ROOT     = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd()
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / 'data' / 'QN_angket_penggunaan_media.csv').exists():
        ROOT = p; break

QN_PATH  = ROOT / 'data' / 'QN_angket_penggunaan_media.csv'
QL_PATH  = ROOT / 'data' / 'QL_angket_penggunaan_media.csv'
POST_PATH = ROOT / 'data' / 'field_test' / 'keterampilan_berbicara_postes.csv'

ITEM_COLS = [f'Q{i}' for i in range(1, 16)]
DIM_COLS  = ['skor_planning', 'skor_monitoring', 'skor_evaluation']
DIM_LABEL = {'skor_planning':'Perencanaan','skor_monitoring':'Pemantauan','skor_evaluation':'Evaluasi'}
DIMS_ITEMS = {
    'Perencanaan': ['Q1','Q4','Q6','Q8','Q15'],
    'Pemantauan' : ['Q5','Q7','Q9','Q10','Q11'],
    'Evaluasi'   : ['Q2','Q3','Q12','Q13','Q14'],
}

def load_qn():
    with open(QN_PATH, encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    out = []
    for row in rows:
        if row.get('kelompok') != 'eksperimen' or not row.get('Q1'):
            continue
        for col in ITEM_COLS + DIM_COLS + ['skor_total']:
            if row.get(col): row[col] = float(row[col])
        out.append(row)
    return out

def load_post():
    with open(POST_PATH, encoding='utf-8-sig', newline='') as f:
        return {r['id']: float(r['post_nilai_akhir']) for r in csv.DictReader(f)}

def load_ql():
    with open(QL_PATH, encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

def describe(x):
    return dict(n=len(x),mean=mean(x),sd=stdev(x) if len(x)>1 else 0,
                median=sorted(x)[len(x)//2],mn=min(x),mx=max(x))

def cronbach(matrix):
    k=len(matrix[0])
    iv=sum(variance([r[j] for r in matrix]) for j in range(k))
    tv=variance([sum(r) for r in matrix])
    return k/(k-1)*(1-iv/tv) if tv else float('nan')

def betacf(a,b,x):
    qab,qap,qam=a+b,a+1,a-1; c,d=1.0,max(abs(1-qab*x/qap),3e-300)
    d=1/d; h=d
    for m in range(1,201):
        m2=2*m; aa=m*(b-m)*x/((qam+m2)*(a+m2))
        d=1+aa*d; d=1/max(abs(d),3e-300)*(1 if d>=0 else -1)
        c=1+aa/c; c=max(abs(c),3e-300)*(1 if c>=0 else -1)
        h*=d*c; aa=-(a+m)*(qab+m)*x/((a+m2)*(qap+m2))
        d=1+aa*d; d=1/max(abs(d),3e-300)*(1 if d>=0 else -1)
        c=1+aa/c; c=max(abs(c),3e-300)*(1 if c>=0 else -1)
        delta=d*c; h*=delta
        if abs(delta-1)<3e-14: break
    return h

def ibeta(a,b,x):
    if x<=0: return 0.0
    if x>=1: return 1.0
    bt=math.exp(math.lgamma(a+b)-math.lgamma(a)-math.lgamma(b)+a*math.log(x)+b*math.log1p(-x))
    return bt*betacf(a,b,x)/a if x<(a+1)/(a+b+2) else 1-bt*betacf(b,a,1-x)/b

def pearson_r(x,y):
    n=len(x); mx,my=mean(x),mean(y)
    num=sum((a-mx)*(b-my) for a,b in zip(x,y))
    den=math.sqrt(sum((a-mx)**2 for a in x)*sum((b-my)**2 for b in y))
    return num/den if den else 0

def p_from_r(r,n):
    if abs(r)>=1: return 0.0
    t=r*math.sqrt(n-2)/math.sqrt(1-r**2)
    return ibeta((n-2)/2,0.5,(n-2)/(n-2+t**2))

qn  = load_qn()
post= load_post()
ql  = load_ql()
n   = len(qn)

# ── statistik ────────────────────────────────────────────────────────────────
dims_stats = {}
for col in DIM_COLS:
    vals = [r[col] for r in qn if isinstance(r.get(col),float)]
    dims_stats[col] = describe(vals)

total_stats = describe([r['skor_total'] for r in qn if isinstance(r.get('skor_total'),float)])

# persentase
sigma_r = sum(sum(r[q] for q in ITEM_COLS) for r in qn)
n_ideal = len(ITEM_COLS)*4*n
p_total = sigma_r/n_ideal*100

dims_p = {}
for dim,items in DIMS_ITEMS.items():
    sr = sum(sum(r[q] for q in items) for r in qn)
    ni = len(items)*4*n
    dims_p[dim] = sr/ni*100

# alpha
alpha_total = cronbach([[r[q] for q in ITEM_COLS] for r in qn])
alpha_dims  = {dim: cronbach([[r[q] for q in items] for r in qn])
               for dim,items in DIMS_ITEMS.items()}

# kategori
cat = Counter(r.get('kategori','') for r in qn)

# korelasi
pairs = [(r['skor_total'],post[r['id']]) for r in qn
         if r['id'] in post and isinstance(r.get('skor_total'),float)]
xs,ys = zip(*pairs)
r_val = pearson_r(list(xs),list(ys))
p_val = p_from_r(r_val,len(pairs))
r_interp = 'kuat' if abs(r_val)>=0.6 else ('sedang' if abs(r_val)>=0.4 else 'lemah')

# ql ringkasan
ql_eks = [r for r in ql if r.get('id','').startswith('E') and r.get('aspek_disukai')]

def interp_alpha(a):
    if a>=0.9: return "sangat baik"
    if a>=0.8: return "baik"
    if a>=0.7: return "dapat diterima"
    if a>=0.6: return "dipertanyakan"
    return "buruk"

OUT_PATH = ROOT / 'data' / 'catatan_pembahasan_angket_media.md'

lines = [
"# Catatan Pembahasan Bab IV — Angket Penggunaan Media\n\n",
"> Catatan ini digenerate otomatis dari notebook `olahdata_angket_media.ipynb`.",
" Gunakan sebagai bahan draf Bab IV bagian E (Respons Mahasiswa) dan F (Temuan Kualitatif).\n\n",
"---\n\n",

"## A. Deskripsi Data Kuantitatif (QN)\n\n",
f"Angket penggunaan media diisi oleh **{n} mahasiswa** kelompok eksperimen menggunakan skala Likert 1–4 ",
"pada 15 butir yang mencakup tiga dimensi: Perencanaan, Pemantauan, dan Evaluasi.\n\n",

"### Tabel: Statistik Deskriptif Skor Angket Penggunaan Media\n\n",
"| Dimensi | n | Mean | SD | Min | Maks |\n",
"|---|---:|---:|---:|---:|---:|\n",
]
for col in DIM_COLS:
    d = dims_stats[col]; lbl = DIM_LABEL[col]
    lines.append(f"| {lbl} | {d['n']} | {d['mean']:.3f} | {d['sd']:.3f} | {d['mn']:.2f} | {d['mx']:.2f} |\n")
d = total_stats
lines.append(f"| **Total** | {d['n']} | **{d['mean']:.3f}** | {d['sd']:.3f} | {d['mn']:.2f} | {d['mx']:.2f} |\n\n")

lines += [
"Rerata skor total angket menunjukkan bahwa secara umum mahasiswa kelompok eksperimen memiliki ",
f"pola penggunaan media yang **{('sangat tinggi' if total_stats['mean']>=3.5 else 'tinggi' if total_stats['mean']>=3 else 'sedang')}** ",
f"(M = {total_stats['mean']:.2f} dari skala 4). ",
f"Dimensi tertinggi adalah **{DIM_LABEL[max(dims_stats, key=lambda k: dims_stats[k]['mean'])]}** ",
f"dan dimensi yang perlu diperhatikan adalah **{DIM_LABEL[min(dims_stats, key=lambda k: dims_stats[k]['mean'])]}**.\n\n",

"### Tabel: Persentase Skor Angket (P = ΣR/N × 100%)\n\n",
"| Dimensi | ΣR | N ideal | P (%) | Kategori |\n",
"|---|---:|---:|---:|---:|\n",
]
for dim,items in DIMS_ITEMS.items():
    p = dims_p[dim]
    cat_p = 'Sangat Tinggi' if p>=85 else ('Tinggi' if p>=70 else ('Sedang' if p>=55 else 'Rendah'))
    sr = sum(sum(r[q] for q in items) for r in qn)
    ni = len(items)*4*n
    lines.append(f"| {dim} | {sr:.0f} | {ni} | {p:.2f}% | {cat_p} |\n")
p_cat = 'Sangat Tinggi' if p_total>=85 else ('Tinggi' if p_total>=70 else ('Sedang' if p_total>=55 else 'Rendah'))
lines.append(f"| **Total** | {sigma_r:.0f} | {n_ideal} | **{p_total:.2f}%** | **{p_cat}** |\n\n")

lines += [
f"Persentase skor total sebesar **{p_total:.2f}%** menempatkan respons mahasiswa dalam kategori **{p_cat}**. ",
"Ini menunjukkan bahwa secara keseluruhan mahasiswa menilai penggunaan media Bicaranta dalam konteks ",
"yang sesuai dengan dimensi-dimensi yang diukur.\n\n",

"### Tabel: Reliabilitas Internal (Alpha Cronbach)\n\n",
"| Dimensi | k butir | Alpha Cronbach | Interpretasi |\n",
"|---|---:|---:|---:|\n",
f"| Total (15 butir) | 15 | {alpha_total:.3f} | {interp_alpha(alpha_total).capitalize()} |\n",
]
for dim,a in alpha_dims.items():
    lines.append(f"| {dim} | 5 | {a:.3f} | {interp_alpha(a).capitalize()} |\n")

lines += [
"\n",
f"Koefisien Alpha Cronbach total sebesar **{alpha_total:.3f}** menunjukkan reliabilitas yang **{interp_alpha(alpha_total)}**. ",
"Angka ini perlu dicermati dalam konteks jumlah butir dan heterogenitas konstruk yang diukur.\n\n",

"### Distribusi Kategori Responden\n\n",
"| Kategori | n | % |\n",
"|---|---:|---:|\n",
]
for k in ['Sangat Tinggi','Tinggi','Sedang','Rendah']:
    v = cat.get(k,0)
    lines.append(f"| {k} | {v} | {v/n*100:.1f}% |\n")

lines += [
"\n",
"Distribusi di atas menunjukkan sebaran kecenderungan mahasiswa dalam penggunaan media. ",
"Mayoritas mahasiswa berada pada kategori tinggi atau sangat tinggi, ",
"yang mengindikasikan bahwa media diterima dan digunakan sesuai intensi perancangan.\n\n",

"---\n\n",
"## B. Korelasi Skor Angket Media × Nilai Postes Berbicara\n\n",
f"Analisis korelasi Pearson dilakukan untuk melihat hubungan antara skor total angket penggunaan media ",
f"dan nilai akhir postes keterampilan berbicara pada kelompok eksperimen (n = {len(pairs)}).\n\n",
f"- **r = {r_val:.3f}**,  p = {p_val:.4f}\n",
f"- Arah dan kekuatan: **{r_interp}**{' dan signifikan' if p_val < 0.05 else ' dan tidak signifikan'} (α = 0,05)\n\n",
]
if p_val < 0.05:
    lines.append(
    f"Korelasi yang **signifikan** (p < 0,05) menunjukkan bahwa terdapat hubungan positif antara "
    f"intensitas dan kualitas penggunaan media dengan performa berbicara mahasiswa. "
    f"Temuan ini konsisten dengan ekspektasi teoretis bahwa media yang digunakan secara aktif "
    f"berkontribusi pada peningkatan keterampilan berbicara (Setyawan \\& Nawangsari, 2021).\n\n"
    )
else:
    lines.append(
    f"Korelasi yang **tidak signifikan** (p ≥ 0,05) menunjukkan bahwa skor angket dan skor berbicara "
    f"tidak bergerak secara linear secara langsung. Ini bisa dijelaskan oleh faktor mediasi lain "
    f"seperti motivasi belajar, frekuensi latihan mandiri, atau variasi kemampuan awal mahasiswa. "
    f"Temuan ini tidak serta merta menunjukkan ketidakefektifan media, melainkan mengisyaratkan "
    f"bahwa hubungan keduanya bersifat tidak langsung dan perlu dijelaskan melalui analisis kualitatif.\n\n"
    )

lines += [
"---\n\n",
"## C. Temuan Kualitatif (QL) — Respons Mahasiswa\n\n",
f"Data kualitatif terbuka diperoleh dari {len(ql_eks)} responden kelompok eksperimen. ",
"Berikut adalah ringkasan respons berdasarkan enam kolom utama.\n\n",
]

QL_FIELDS = {
    'aspek_disukai':        'Aspek yang paling disukai',
    'alasan_aspek_disukai': 'Alasan menyukai aspek tersebut',
    'manfaat':              'Manfaat yang dirasakan',
    'kendala':              'Kendala penggunaan',
    'saran_perbaikan':      'Saran perbaikan',
    'rekomendasi':          'Rekomendasi',
}
for field, label in QL_FIELDS.items():
    resps = [r[field] for r in ql_eks if r.get(field) and len(r[field])>3]
    if not resps: continue
    lines.append(f"### {label}\n\n")
    for i, resp in enumerate(resps[:4], 1):
        clean = resp.replace('"', '\\"')
        lines.append(f'> [{i}] "{clean}"\n\n')

lines += [
"---\n\n",
"## D. Ringkasan Integratif untuk Pembahasan Bab IV\n\n",
"Berdasarkan data kuantitatif (QN) dan kualitatif (QL) dari angket penggunaan media, ",
"dapat dirumuskan beberapa poin pembahasan berikut:\n\n",
f"1. **Tingkat penggunaan media**: Mahasiswa kelompok eksperimen menunjukkan pola penggunaan media "
f"yang secara rata-rata berada dalam kategori **{p_cat.lower()}** (P = {p_total:.2f}%). "
f"Hal ini mengindikasikan bahwa media Bicaranta digunakan sesuai dengan maksud perancangannya.\n\n",
f"2. **Dimensi terkuat**: Dimensi **{DIM_LABEL[max(dims_stats, key=lambda k: dims_stats[k]['mean'])]}** "
f"(M = {dims_stats[max(dims_stats, key=lambda k: dims_stats[k]['mean'])]['mean']:.3f}) "
f"merupakan aspek yang paling kuat dipersepsi mahasiswa, menunjukkan bahwa media efektif dalam mendorong "
f"mahasiswa untuk merencanakan dan memantau proses belajar mereka.\n\n",
f"3. **Reliabilitas**: Alpha Cronbach total sebesar {alpha_total:.3f} perlu dibahas secara kritis; "
f"nilai yang {'cukup tinggi' if alpha_total >= 0.7 else 'relatif rendah'} ini "
f"{'menunjukkan konsistensi internal yang memadai' if alpha_total >= 0.7 else 'mengisyaratkan heterogenitas antar butir yang perlu dikaji lebih lanjut'}.\n\n",
"4. **Korelasi dan temuan QL**: Respons mahasiswa secara kualitatif menggambarkan "
"manfaat, kendala, dan saran yang memperkaya interpretasi data angket. "
"Kutipan-kutipan di atas dapat digunakan sebagai bukti penunjang dalam pembahasan Bab IV "
"bagian F (Temuan Kualitatif) dan G (Integrasi Mixed Methods).\n\n",
"---\n",
f"*File ini digenerate dari `olahdata_angket_media.ipynb` — {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M')}*\n",
]

OUT_PATH.write_text("".join(lines), encoding='utf-8')
print(f"OK  {OUT_PATH}")
print(f"    n={n}, P_total={p_total:.2f}%, alpha={alpha_total:.3f}, r={r_val:.3f} (p={p_val:.4f})")


OK  /home/primandhika/artikel/dist/data/catatan_pembahasan_angket_media.md
    n=40, P_total=62.25%, alpha=-0.081, r=0.214 (p=0.1857)
